# 2 — Feature Engineering

This notebook demonstrates the full feature-engineering pipeline,
showing how raw LMP + weather data is transformed into a model-ready
feature matrix.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pjm_spike_forecast.data import build_demo_dataset
from pjm_spike_forecast.config import FeatureConfig
from pjm_spike_forecast.features import (
    add_temporal_features,
    add_lag_features,
    add_rolling_features,
    add_weather_features,
    label_spikes,
    build_feature_matrix,
    get_feature_columns,
)

df = build_demo_dataset(n_days=365, seed=42)
print(f"Raw data: {df.shape}")
df.head()

## 2.1 — Temporal Features

Cyclical encoding of hour and month prevents the model from
treating hour 23 and hour 0 as maximally distant.

In [ ]:
df_temp = add_temporal_features(df)
print("New columns:", [c for c in df_temp.columns if c not in df.columns])

# Show cyclical encoding
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(df_temp["hour_cos"], df_temp["hour_sin"], c=df_temp["hour"],
                cmap="hsv", s=5, alpha=0.3)
axes[0].set_xlabel("hour_cos"); axes[0].set_ylabel("hour_sin")
axes[0].set_title("Cyclical Hour Encoding")

axes[1].scatter(df_temp["month_cos"], df_temp["month_sin"], c=df_temp["month"],
                cmap="hsv", s=5, alpha=0.3)
axes[1].set_xlabel("month_cos"); axes[1].set_ylabel("month_sin")
axes[1].set_title("Cyclical Month Encoding")
plt.tight_layout()
plt.show()

## 2.2 — Lag & Rolling Features

In [ ]:
df_lag = add_lag_features(df, "lmp", lags=[1, 24, 168])
print("Lag columns:", [c for c in df_lag.columns if "lag" in c])

df_roll = add_rolling_features(df, "lmp", windows=[24, 168])
print("Rolling columns:", [c for c in df_roll.columns if "rmean" in c or "rstd" in c])

# Plot: raw LMP vs 24h rolling mean
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_roll.index[-500:], df_roll["lmp"].iloc[-500:], alpha=0.5, label="Raw LMP")
ax.plot(df_roll.index[-500:], df_roll["lmp_rmean_24"].iloc[-500:], color="red", label="24h Rolling Mean")
ax.set_ylabel("$/MWh"); ax.set_title("LMP vs 24h Rolling Mean"); ax.legend()
plt.tight_layout()
plt.show()

## 2.3 — Spike Labelling

A spike is an hour where LMP exceeds the 1-week rolling mean + 2σ.

In [ ]:
df_spike = label_spikes(df, window=168, n_std=2.0)
valid = df_spike.dropna()
spike_pct = valid["spike"].mean() * 100
print(f"Spike rate: {spike_pct:.1f}% of hours")
print(f"Number of spikes: {int(valid['spike'].sum())} out of {len(valid)} hours")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(valid.index, valid["lmp"], linewidth=0.4, color="steelblue")
ax.plot(valid.index, valid["spike_threshold"], linewidth=0.5, color="orange", label="Threshold")
spikes = valid[valid["spike"] == 1]
ax.scatter(spikes.index, spikes["lmp"], color="red", s=8, zorder=5, label="Spike")
ax.set_ylabel("$/MWh"); ax.set_title("Spike Detection"); ax.legend()
plt.tight_layout()
plt.show()

## 2.4 — Full Feature Matrix

In [ ]:
feat_df = build_feature_matrix(df)
feature_cols = get_feature_columns(feat_df)
print(f"Feature matrix: {feat_df.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"No NaN values: {feat_df.isna().sum().sum() == 0}")
print(f"\nFeature list:")
for i, c in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {c}")